# Query Gym 

This notebook aims to illustrate the use of QueryGym Environment. This demo focuses only on the text2sql benchmark. Specifically the Bird Dataset.



### 1. First some imports

In [ ]:
from query_gym.dataset.unite_dataset import UniteDataset
from query_gym.envs import SQLiteQueryGym as QueryGym
from dotenv import find_dotenv, load_dotenv
from pprint import pprint
load_dotenv(find_dotenv())

import os
# Assert LITELLM_API_KEY is available
assert os.environ.get("LITELLM_API_KEY")


### 2. Load the Dataset

In [3]:
dataset = UniteDataset(dataset_path=f"/data/datasets/unified/bird_train",mode="train")
pprint(dataset[0])

DbDatasetItem(path='file:/data/datasets/unified/bird_train/database/movie_platform/movie_platform.sqlite?mode=ro',
              gt='SELECT movie_title FROM movies WHERE movie_release_year = '
                 '1945 ORDER BY movie_popularity DESC LIMIT 1',
              query='Name movie titles released in year 1945. Sort the listing '
                    'by the descending order of movie popularity.',
              info={'SQL': 'SELECT movie_title FROM movies WHERE '
                           'movie_release_year = 1945 ORDER BY '
                           'movie_popularity DESC LIMIT 1',
                    'db_id': 'movie_platform',
                    'evidence': 'released in the year 1945 refers to '
                                'movie_release_year = 1945;',
                    'query': 'SELECT movie_title FROM movies WHERE '
                             'movie_release_year = 1945 ORDER BY '
                             'movie_popularity DESC LIMIT 1',
                    'que

### 3. Initiate the environment

A Gym Environment (now part of Gymnasium) is a standardized API for interacting with environments in Reinforcement Learning (RL). It abstracts away the complexity of different domains---games, robotics, simulations---into a consistent interface so agents can be trained and evaluated without worrying about environment-specific details.


Example: 

<div style="background-color: lightgrey; padding: 10px; border-radius: 5px;">

```python
import CartPole
 
# Create the environment
env = CartPole()

# Reset env and get first observation
obs = env.reset()
 
# Step in the env with random actions
# and display the env
for _ in range(100):
    # Display the env
    env.render(mode="human")
 
    action = env.action_space.sample()
    # Retrieve new observation, reward, termination signal
    # and additional infos
    obs, reward, done, truncated, info = env.step(action)
 
    # End of an episode
    if done or truncated:
        obs = env.reset()

```    
</div>


**Core Concepts:** 
* env = CartPole()
Creates an instance of the CartPole environment.
* obs = env.reset()
Resets the environment to an new initial state and returns the first observation.
* env.step(action)
Takes an action and returns:
    * obs: New observation (state)
    * reward: Reward for the action
    * done: Boolean indicating if the goal is achieved
    * truncated: Boolean indicating if the episode ended
    * info: Extra diagnostic info

In [4]:
env = QueryGym(dataset=dataset)
obs = env.reset(2692)


In [5]:
print(obs)

The database `regional_sales` contains following tables: Customers,Products,Regions,Sales Team,Store Locations,Sales Orders. 
Table `Customers` contains following columns: `CustomerID,Customer Names`. 
Table `Products` contains following columns: `ProductID,Product Name`. 
Table `Regions` contains following columns: `StateCode,State,Region`. 
Table `Sales Team` contains following columns: `SalesTeamID,Sales Team,Region`. 
Table `Store Locations` contains following columns: `StoreID,City Name,County,StateCode,State,Type,Latitude,Longitude,AreaCode,Population,Household Income,Median Income,Land Area,Water Area,Time Zone`. 
Table `Sales Orders` contains following columns: `OrderNumber,Sales Channel,WarehouseCode,ProcuredDate,OrderDate,ShipDate,DeliveryDate,CurrencyCode,_SalesTeamID,_CustomerID,_StoreID,_ProductID,Order Quantity,Discount Applied,Unit Price,Unit Cost`. 
User Query: In which city is the store with the highest sales order unit price located?


### 4. Take a manual step in the env

In [6]:
obs, reward, done, truncated, info  =  env.step(["afdsd"])

In [7]:
print(obs)

Unrecognized action. There are 20 types of actions in this scenario. The action name, description, usage, and example is given below.
1. get_overview: {'description': 'Get an overview of the task.', 'usage': 'Action: ["get_overview"]\n'}
2. get_query: {'description': 'Get the user query.', 'usage': 'Action: ["get_query"]\n'}
3. get_schema: {'description': 'Get the schema.', 'usage': 'Action: ["get_schema"]\n'}
4. get_tables: {'description': 'Get a list of tables in the database.', 'usage': 'Action: ["get_tables"]\n'}
5. get_columns: {'description': 'Get a list of columns in a given table.', 'usage': 'Action: ["get_columns", "<table_id>"]\n', 'examples_description': 'Example #1: ["get_columns", "T_0"] Example #2: ["get_columns", "frpm"]\n'}
6. get_actions: {'description': 'Get a list of actions.', 'usage': 'Action: ["get_actions"]\n'}
7. get_operations: {'description': 'Get a list of operations.', 'usage': 'Action: ["get_operations"]\n'}
8. perform_projection: {'description': 'Project c

In [8]:
obs, reward, done, truncated, info  =  env.step(["get_schema"])

In [9]:
print(obs)

## Schema

CREATE TABLE Customers
(
    CustomerID       INTEGER
        constraint Customers_pk
            primary key,
    "Customer Names" TEXT
)

CREATE TABLE Products
(
    ProductID      INTEGER
        constraint Products_pk
            primary key,
    "Product Name" TEXT
)

CREATE TABLE Regions
(
    StateCode TEXT
        constraint Regions_pk
            primary key,
    State     TEXT,
    Region    TEXT
)

CREATE TABLE "Sales Team"
(
    SalesTeamID  INTEGER
        constraint "Sales Team_pk"
            primary key,
    "Sales Team" TEXT,
    Region       TEXT
)

CREATE TABLE "Store Locations"
(
    StoreID            INTEGER
        constraint "Store Locations_pk"
            primary key,
    "City Name"        TEXT,
    County             TEXT,
    StateCode          TEXT
        constraint "Store Locations_Regions_StateCode_fk"
            references Regions(StateCode),
    State              TEXT,
    Type               TEXT,
    Latitude           REAL,
    Longitud

### 6. What is in the info?

In [10]:
pprint(info)

{'GT': 'SELECT T2.`City Name` FROM `Sales Orders` AS T1 INNER JOIN `Store '
       'Locations` AS T2 ON T2.StoreID = T1._StoreID WHERE REPLACE(T1.`Unit '
       "Price`, ',', '') = ( SELECT REPLACE(T1.`Unit Price`, ',', '') FROM "
       '`Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID '
       "= T1._StoreID ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1 "
       ") ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1",
 'SQL': 'SELECT T2.`City Name` FROM `Sales Orders` AS T1 INNER JOIN `Store '
        'Locations` AS T2 ON T2.StoreID = T1._StoreID WHERE REPLACE(T1.`Unit '
        "Price`, ',', '') = ( SELECT REPLACE(T1.`Unit Price`, ',', '') FROM "
        '`Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID '
        "= T1._StoreID ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1 "
        ") ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1",
 'db_id': 'regional_sales',
 'evidence': 'highest sales order unit price ref

In [11]:
obs, reward, done, truncated, info  =  env.step(["perform_order_by", '`Sales Orders`', '`Unit Price` DESC','`_StoreID`, `Unit Price`'])

In [12]:
pprint(info['steps']) 
pprint(info['generated_sql'])
pprint(info['trace'])

3
'SELECT `_StoreID`, `Unit Price` FROM `Sales Orders` ORDER BY `Unit Price` DESC'
{'T_0': 'SELECT `_StoreID`, `Unit Price` FROM `Sales Orders` ORDER BY `Unit '
        'Price` DESC'}


In [13]:
print(obs)

Table T_0 has 2 column(s) and 7991 row(s).
Column names are _StoreID,Unit Price.



In [14]:
obs, reward, done, truncated, info  =  env.step(["perform_limit", 'T_0', '1'])

In [15]:
print(obs)
pprint(info)

Table T_1 has 2 column(s) and 1 row(s).
Column names are _StoreID,Unit Price.

{'GT': 'SELECT T2.`City Name` FROM `Sales Orders` AS T1 INNER JOIN `Store '
       'Locations` AS T2 ON T2.StoreID = T1._StoreID WHERE REPLACE(T1.`Unit '
       "Price`, ',', '') = ( SELECT REPLACE(T1.`Unit Price`, ',', '') FROM "
       '`Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID '
       "= T1._StoreID ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1 "
       ") ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1",
 'SQL': 'SELECT T2.`City Name` FROM `Sales Orders` AS T1 INNER JOIN `Store '
        'Locations` AS T2 ON T2.StoreID = T1._StoreID WHERE REPLACE(T1.`Unit '
        "Price`, ',', '') = ( SELECT REPLACE(T1.`Unit Price`, ',', '') FROM "
        '`Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID '
        "= T1._StoreID ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1 "
        ") ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1

In [16]:
obs, reward, done, truncated, info  =  env.step(['perform_join',  ['T_1' , "`Store Locations`"], ["T_1._StoreID = `Store Locations`.StoreID"], ["inner join"] , '`City Name`'])

In [17]:
print(obs)
print(reward)
pprint(info['generated_sql'])

Resulting table, T_2, has single value of Renton
100
('WITH T_0 as (SELECT `_StoreID`, `Unit Price` FROM `Sales Orders` ORDER BY '
 '`Unit Price` DESC),T_1 as (SELECT * FROM T_0 LIMIT 1) SELECT `City Name` '
 'FROM T_1 inner join `Store Locations` ON T_1._StoreID = `Store '
 'Locations`.StoreID')


### 5. Env with SQL Execution Option

In [18]:
env = QueryGym(dataset=dataset, allow_execute_sql=True)

In [19]:
t = env.reset(2692)
obs, reward, done, truncated, info  =  env.step(["afdsd"])

In [20]:
print(obs)

Unrecognized action. There are 21 types of actions in this scenario. The action name, description, usage, and example is given below.
1. execute_sql: {'description': 'Execute the SQL command.', 'usage': 'Action: ["execute_sql", "<query>"]\n', 'examples_description': 'Example #1: ["execute_sql", "Select * from employees"]\n'}
2. get_overview: {'description': 'Get an overview of the task.', 'usage': 'Action: ["get_overview"]\n'}
3. get_query: {'description': 'Get the user query.', 'usage': 'Action: ["get_query"]\n'}
4. get_schema: {'description': 'Get the schema.', 'usage': 'Action: ["get_schema"]\n'}
5. get_tables: {'description': 'Get a list of tables in the database.', 'usage': 'Action: ["get_tables"]\n'}
6. get_columns: {'description': 'Get a list of columns in a given table.', 'usage': 'Action: ["get_columns", "<table_id>"]\n', 'examples_description': 'Example #1: ["get_columns", "T_0"] Example #2: ["get_columns", "frpm"]\n'}
7. get_actions: {'description': 'Get a list of actions.',

In [21]:
obs, reward, done, truncated, info  =  env.step(["execute_sql", "SELECT `_StoreID`, `Unit Price` FROM `Sales Orders` ORDER BY `Unit Price` DESC limit 1"])

In [22]:
print(obs)

Table T_0 has 2 column(s) and 1 row(s).
Column names are _StoreID,Unit Price.



In [23]:
obs, reward, done, truncated, info  =  env.step(["execute_sql", "SELECT `City Name` FROM T_0 inner join `Store Locations` ON T_0._StoreID = `Store Locations`.StoreID"])

In [24]:
print(obs)
print(reward)
pprint(info['generated_sql'])

Resulting table, T_1, has single value of Renton
100
('WITH T_0 as (SELECT `_StoreID`, `Unit Price` FROM `Sales Orders` ORDER BY '
 '`Unit Price` DESC limit 1) SELECT `City Name` FROM T_0 inner join `Store '
 'Locations` ON T_0._StoreID = `Store Locations`.StoreID')


### 6. Use Env with Open AI Chat for Text2SQL


Initialize a client 

In [ ]:
from openai import OpenAI

BASE_URL = "http://0.0.0.0:4000/llama-3-3-70b-instruct/v1"
MODEL= "meta-llama/llama-3-3-70b-instruct"



client = OpenAI(api_key=os.environ["LITELLM_API_KEY"],
                base_url=BASE_URL,
                default_headers={}
                )

def invoke_llm(messages):
    res = client.chat.completions.create(
            model=MODEL,
            messages= messages,
            n=1,
            temperature=0.7,
            stop="Input", max_tokens =6000
            )
    pprint(res)
    return res.choices[0].message.content

Setup Environment and message

In [35]:

obs = env.reset(2692)

system_instruction = """You are an expert SQL Programming assistant. Always return your SQL as follows:
```sql
...
```"""


messages = [{"role": "system", "content": system_instruction },
    {"role": "user", "content": obs}]

In [27]:
res = invoke_llm(messages)

ChatCompletion(id='chatcmpl-1e2c67b9-bb88-4f29-be0a-6fec32e28fd4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='To find the city with the highest sales order unit price, we need to join the `Sales Orders` table with the `Store Locations` table and then find the maximum unit price. Here is the SQL query to achieve this:\n\n```sql\nSELECT T2.CityName\nFROM SalesOrders AS T1\nINNER JOIN `Store Locations` AS T2\nON T1._StoreID = T2.StoreID\nORDER BY T1.UnitPrice DESC\nLIMIT 1;\n```\n\nHowever, the above query will only return one row even if there are multiple stores with the same highest unit price. If you want to get all the cities with the highest unit price, you can use the following query:\n\n```sql\nSELECT T2.CityName\nFROM SalesOrders AS T1\nINNER JOIN `Store Locations` AS T2\nON T1._StoreID = T2.StoreID\nWHERE T1.UnitPrice = (SELECT MAX(UnitPrice) FROM SalesOrders);\n```\n\nThis query will return all the cities where the store

In [28]:
def extract_sql(content):
    return content.split("```sql")[1].split("```")[0].replace("```sql", "").replace("```", "").replace(";", "").strip()
                

In [29]:
sql = extract_sql(res)

In [30]:
print(sql)

SELECT T2.CityName
FROM SalesOrders AS T1
INNER JOIN `Store Locations` AS T2
ON T1._StoreID = T2.StoreID
ORDER BY T1.UnitPrice DESC
LIMIT 1


In [31]:
obs, reward, done, truncated, info  =   env.step(['execute_sql', sql])

In [32]:
print(obs)

Execute SQL failed.


# Exception

```
Traceback (most recent call last):
  File "/data/projects/QueryPlanning/QueryGym/query_gym/envs/query_gym_env.py", line 209, in step
    table_id_or_msg = self.execute_query_with_cte(action_params[0])
  File "/data/projects/QueryPlanning/QueryGym/query_gym/envs/query_gym_sqlite.py", line 46, in execute_query_with_cte
    res, description = self.execute_query( execute_query)
                       ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/data/projects/QueryPlanning/QueryGym/query_gym/envs/query_gym_sqlite.py", line 37, in execute_query
    return execute_sql_query(self.conn, query)
  File "/data/projects/QueryPlanning/QueryGym/query_gym/db_utils/sqlite_utils.py", line 88, in execute_sql_query
    cursor.execute(query)
    ~~~~~~~~~~~~~~^^^^^^^
sqlite3.OperationalError: no such table: SalesOrders

```



### 7. Use env with Langchain GymAgent

In [ ]:
from agents.gym_agent import GymnasiumAgent
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI( model=MODEL,
            temperature=0,
            max_retries=3,
            api_key=os.environ["LITELLM_API_KEY"],
            base_url=BASE_URL,
            default_headers={},
        )
agent = GymnasiumAgent(
        model=chat_model,
        instruction= system_instruction,  # System-level instruction for the agent
        max_tokens_input=131072,  # Max tokens for input context
    )

debug_instruction = """The suggested SQL gave following error. 

```
{traceback}
```

So, please fix the SQL. 

Note: Always return your SQL as follows:
```sql
...
```"""

parse_instruction = """Error in parsing the SQL. 

Please only return the SQL as follows:
```sql
...
```"""

Some Error in Git: /data/projects/QueryPlanning/QueryGym
Traceback (most recent call last):
  File "/data/projects/QueryPlanning/QueryGym/query_gym/utils/logging_util.py", line 1416, in get_git_infos
    repo = git.Repo(directory)
  File "/data/miniforge3/envs/qgym_env/lib/python3.13/site-packages/git/repo/base.py", line 289, in __init__
    raise InvalidGitRepositoryError(epath)
git.exc.InvalidGitRepositoryError: /data/projects/QueryPlanning/QueryGym



In [47]:
import traceback
agent.reset()
obs = env.reset(2692)

agent.observe(obs)

for _ in range(3):
    try: 
        action_str = agent.act()
        pred_sql = extract_sql(action_str)
        print(pred_sql)
        try:
            observation, reward, done, truncated, info = env.step(["execute_sql", pred_sql])
            print(observation)
            if reward == env.GOAL_REWARD:
                print("The user query is answered.\n")
            elif reward == env.STEP_REWARD:
                print("The user query is almost answered. Keep Going.\n")

        except Exception as e:
            # Handle execution errors
            observation = debug_instruction.replace("{traceback}", traceback.format_exc())
            
    except  Exception as e:
        # Handle parsing errors
        observation = parse_instruction.replace("{traceback}", traceback.format_exc())
    
    # Stop if query is answered or episode is truncated
    if done or truncated:
        if done:
            print("done ")
        if truncated:
            print('truncated')
        break
    agent.observe(observation)
        

SELECT T2.CityName
FROM SalesOrders AS T1
INNER JOIN StoreLocations AS T2
ON T1._StoreID = T2.StoreID
ORDER BY T1.UnitPrice DESC
LIMIT 1
Execute SQL failed.


# Exception

```
Traceback (most recent call last):
  File "/data/projects/QueryPlanning/QueryGym/query_gym/envs/query_gym_env.py", line 209, in step
    table_id_or_msg = self.execute_query_with_cte(action_params[0])
  File "/data/projects/QueryPlanning/QueryGym/query_gym/envs/query_gym_sqlite.py", line 46, in execute_query_with_cte
    res, description = self.execute_query( execute_query)
                       ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/data/projects/QueryPlanning/QueryGym/query_gym/envs/query_gym_sqlite.py", line 37, in execute_query
    return execute_sql_query(self.conn, query)
  File "/data/projects/QueryPlanning/QueryGym/query_gym/db_utils/sqlite_utils.py", line 88, in execute_sql_query
    cursor.execute(query)
    ~~~~~~~~~~~~~~^^^^^^^
sqlite3.OperationalError: no such table: SalesOrders

```

SELECT T2

In [49]:
agent.message_history


[AIMessage(content='To find the city where the store with the highest sales order unit price is located, we need to join the `Sales Orders` table with the `Store Locations` table and then find the maximum unit price. Here is the SQL query to achieve this:\n\n```sql\nSELECT T2.CityName\nFROM SalesOrders AS T1\nINNER JOIN StoreLocations AS T2\nON T1._StoreID = T2.StoreID\nORDER BY T1.UnitPrice DESC\nLIMIT 1;\n```\n\nThis query works as follows:\n- It selects the `CityName` from the `StoreLocations` table.\n- It joins the `SalesOrders` table with the `StoreLocations` table based on the store ID.\n- It orders the results in descending order based on the `UnitPrice` in the `SalesOrders` table.\n- It limits the output to the top row, which corresponds to the store with the highest sales order unit price.\n\nHowever, if there are multiple stores with the same highest unit price, this query will only return one of them. If you want to get all stores with the highest unit price, you can use a s

In [46]:
observation, reward, done, truncated, info


('Resulting table, T_0, has single value of Renton',
 100,
 True,
 False,
 {'db_id': 'regional_sales',
  'question': 'In which city is the store with the highest sales order unit price located?',
  'evidence': 'highest sales order unit price refers to Max(Unit Price)',
  'SQL': "SELECT T2.`City Name` FROM `Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID = T1._StoreID WHERE REPLACE(T1.`Unit Price`, ',', '') = ( SELECT REPLACE(T1.`Unit Price`, ',', '') FROM `Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID = T1._StoreID ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1 ) ORDER BY REPLACE(T1.`Unit Price`, ',', '') DESC LIMIT 1",
  'query': "SELECT T2.`City Name` FROM `Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID = T1._StoreID WHERE REPLACE(T1.`Unit Price`, ',', '') = ( SELECT REPLACE(T1.`Unit Price`, ',', '') FROM `Sales Orders` AS T1 INNER JOIN `Store Locations` AS T2 ON T2.StoreID = T1._StoreID ORDER BY REPLACE(T1.`U